# 3.2 누적 보상, 할인율 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter03_2_discounted_return.ipynb)

책 본문: [3.2 누적 보상, 할인율, 그리고 실습](https://smhanlab.com/book-ml/kor/ml2/chapter03/2.html)

이 노트북은 3.2절의 핵심 — **할인된 리턴** \\(G_t\\)과 **할인율** \\(\\gamma\\) — 을
숫자로 확인합니다:

1. **닫힌 형태 vs 직접 누적**: CartPole의 10스텝 에피소드에서
   \\(G_0 = 8.025\\)가 실제로 누적되는 것을 확인.
2. **기하급수 수렴**: 매 스텝 보상 1인 환경에서
   \\(G_N = \\frac{1-\\gamma^N}{1-\\gamma}\\)가 본문 표의 값(4.10, 6.51, …)으로
   수렴하는 것을 확인.
3. **유효 시계**: \\(H = \\frac{\\log \\epsilon}{\\log \\gamma}\\)를 계산해
   "할인율 0.95 = 약 58스텝짜리 시야"가 나오는 것을 확인.
4. **"할인율은 목표를 바꾼다"**: 본문 예 2의 3상태 MDP에서 정책 A·B의
   상태 0 가치를 \\(\\gamma\\) 함수로 그려, **크로스오버 \\(\\gamma^* \\approx 0.27\\)**
   를 직접 찾는다.
5. **CartPole 무작위 정책 베이스라인**: 같은 환경에서 \\(\\gamma\\)가
   평균 리턴을 어떻게 바꾸는지(8.62 → 13.07 → 20.77).
6. **그림 2장** 생성: 본문에 삽입된 3패널 개요 그림과
   "할인율 vs 정책 선호" 그래프.


## 0. 환경 준비

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

import os
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)
print(f"그림 저장 위치: {IMG}")


numpy 2.4.6 | matplotlib 3.11.1
그림 저장 위치: /home/smhan/book-ml/kor/src/images


## 1. 닫힌 형태 vs 직접 누적 — CartPole 10스텝 에피소드

CartPole은 매 스텝 살아 있으면 \\(+1\\)을 준다. 따라서 10스텝 모두
살아남으면 \\(G_0 = 1 + 0.95 + 0.95^2 + \\cdots + 0.95^9\\).
루프로 **직접 누적**하고, **닫힌 형태**
\\(\\frac{1-0.95^{10}}{1-0.95}\\)로 계산한 값이 같은지 확인한다.
(시드 0이면 10스텝 모두 생존한다 — 터미널이 아닌 한, 할인 계수는
보상이 0이 되는 스텝까지 계속 곱해진다.)

In [ ]:
import gymnasium as gym

env = gym.make("CartPole-v1")
obs, info = env.reset(seed=0)
env.action_space.seed(0)   # 시드 0이면 10스텝 모두 생존 (결정적으로 재현)

gamma = 0.95
total, discount = 0.0, 1.0
print(" 스텝 |  보상  |  계수 γ^t  |  누적 G_0")
print("-" * 46)
for t in range(10):
    a = env.action_space.sample()
    obs, r, term, trunc, _ = env.step(a)
    total += discount * r
    print(f"  {t:2d} |  {r:4.0f}  |   {discount:6.4f}  |  {total:7.4f}")
    discount *= gamma
    if term or trunc:
        print("  (여기서 에피소드 종료)")
        break

closed = (1 - gamma**10) / (1 - gamma)
print(f"\n직접 누적 G_0 = {total:.4f}")
print(f"닫힌 형태       = {closed:.4f}")
assert abs(total - closed) < 1e-9
print("[OK] 직접 누적과 닫힌 형태가 정확히 일치 (본문의 8.025)")
env.close()


 스텝 |  보상  |  계수 γ^t  |  누적 G_0
----------------------------------------------
   0 |     1  |   1.0000  |   1.0000
   1 |     1  |   0.9500  |   1.9500
   2 |     1  |   0.9025  |   2.8525
   3 |     1  |   0.8574  |   3.7099
   4 |     1  |   0.8145  |   4.5244
   5 |     1  |   0.7738  |   5.2982
   6 |     1  |   0.7351  |   6.0333
   7 |     1  |   0.6983  |   6.7316
   8 |     1  |   0.6634  |   7.3950
   9 |     1  |   0.6302  |   8.0253

직접 누적 G_0 = 8.0253
닫힌 형태       = 8.0253
[OK] 직접 누적과 닫힌 형태가 정확히 일치 (본문의 8.025)


## 2. 기하급수 수렴: 본문의 표를 코드로 재확인

매 스텝 보상 \\(R=1\\)인 환경에서 \\(G_N = \\sum_{k=0}^{N-1} \\gamma^k
= \\frac{1-\\gamma^N}{1-\\gamma}\\). 본문의 표
(\\(N \\in \\{5,10,20,50,100\\}\\), \\(\\gamma \\in \\{0.9, 0.99\\}\\))를
코드로 계산해 대조한다. 

In [ ]:
N_list  = [5, 10, 20, 50, 100]
G_table = {}
for g in [0.9, 0.99]:
    for N in N_list:
        G_table[(g, N)] = (1 - g**N) / (1 - g)

print(f"{'N':>4} | {'γ=0.9':>8} | {'γ=0.99':>8}")
print("-" * 28)
for N in N_list:
    print(f"{N:4d} | {G_table[(0.9, N)]:8.4f} | {G_table[(0.99, N)]:8.4f}")
print(f"{'∞':>4} | {1/(1-0.9):8.4f} | {1/(1-0.99):8.4f}")

# 본문 표의 값과 대조
expected = {(0.9, 5): 4.10, (0.9, 10): 6.51, (0.9, 20): 8.78, (0.9, 50): 9.95,
            (0.99, 5): 4.90, (0.99, 10): 9.56, (0.99, 20): 18.21, (0.99, 50): 39.50}
for (g, N), e in expected.items():
    assert abs(round(G_table[(g, N)], 2) - e) < 5e-3, (g, N)
print("\n[OK] 본문 표의 값과 일치 (반올림 오차 범위 내)")


   N |    γ=0.9 |   γ=0.99
----------------------------
   5 |   4.0951 |   4.9010
  10 |   6.5132 |   9.5618
  20 |   8.7842 |  18.2093
  50 |   9.9485 |  39.4994
 100 |   9.9997 |  63.3968
   ∞ |  10.0000 | 100.0000

[OK] 본문 표의 값과 일치 (반올림 오차 범위 내)


## 3. 유효 시계: \\(H = \\frac{\\log \\epsilon}{\\log \\gamma}\\)

"\\(\\gamma^k\\)가 \\(\\epsilon\\) 아래로 *떨어지는* 첫 스텝"을 \\(H\\)라 하면
\\(\\gamma^H = \\epsilon\\)이므로 \\(H = \\frac{\\log \\epsilon}{\\log \\gamma}\\).
\\(\\epsilon = 0.05\\)로 잡고 본문 표를 재확인한다. 확인 문제 1처럼
\\(0.9^{28} < 0.05 < 0.9^{27}\\)를 직접 곱셈으로도 검증한다. 

In [ ]:
eps = 0.05
import math
print(f"{'γ':>6} | {'H = log(0.05)/log(γ)':>22} |  검증: γ^floor(H) vs γ^ceil(H)")
print("-" * 66)
for g in [0.5, 0.9, 0.95, 0.99]:
    H = np.log(eps) / np.log(g)
    k0, k1 = math.floor(H), math.ceil(H)
    print(f"{g:6.2f} | {H:22.2f} |  {g}^{k0}={g**k0:.4f}  {g}^{k1}={g**k1:.4f}")

# 확인 문제 1: 로그 없이 직접 곱셈으로 검증
# 0.9^28 = 0.0523 (5%를 *조금* 위), 0.9^29 = 0.0471 (5% 아래)
assert 0.9**28 > 0.05 > 0.9**29
print(f"\n직접 곱셈: 0.9^28 = {0.9**28:.4f} (5% 살짝 위),  0.9^29 = {0.9**29:.4f} (5% 아래)")
print("[OK] 5% 아래로 떨어지는 첫 정수 스텝 = 29, 실수값 H = 28.43 ≈ 28 (본문: 약 28스텝)")
print("[OK] γ=0.9의 유효 시계는 28스텝 (본문: 약 28스텝)")

# 반대로 쓰는 법: γ = ε^(1/H)
print(f"\n500스텝을 다 보려면 γ = 0.05^(1/500) = {0.05**(1/500):.4f}")
print(f"100스텝을 다 보려면 γ = 0.05^(1/100) = {0.05**(1/100):.4f}")


     γ |   H = log(0.05)/log(γ) |  검증: γ^floor(H) vs γ^ceil(H)
------------------------------------------------------------------
  0.50 |                   4.32 |  0.5^4=0.0625  0.5^5=0.0312
  0.90 |                  28.43 |  0.9^28=0.0523  0.9^29=0.0471
  0.95 |                  58.40 |  0.95^58=0.0510  0.95^59=0.0485
  0.99 |                 298.07 |  0.99^298=0.0500  0.99^299=0.0495

직접 곱셈: 0.9^28 = 0.0523 (5% 살짝 위),  0.9^29 = 0.0471 (5% 아래)
[OK] 5% 아래로 떨어지는 첫 정수 스텝 = 29, 실수값 H = 28.43 ≈ 28 (본문: 약 28스텝)
[OK] γ=0.9의 유효 시계는 28스텝 (본문: 약 28스텝)

500스텝을 다 보려면 γ = 0.05^(1/500) = 0.9940
100스텝을 다 보려면 γ = 0.05^(1/100) = 0.9705


## 4. 예 2: 3상태 MDP — 할인율이 "최적 정책"을 뒤집는다

본문 예 2의 환경:

| 현재 상태 | 행동 A | 행동 B |
|---|---|---|
| 0 | → 1, 보상 +1 | → 1, 보상 +1 |
| 1 | → 2(50%), 0(50%), 보상 +2 | → 0(확정), 보상 0 |
| 2 (루프) | 자기 자신으로, 보상 −10 | 자기 자신으로, 보상 −10 |

상태 2는 "터미널"이 아니라 **−10을 계속 주는 루프**다(터미널이면
\\(V(2)=0\\)이 되어 구덩이의 할인 효과를 볼 수 없다).
두 정책의 상태 0 가치를 \\(\\gamma\\)에 대한 **닫힌 형태**로 써본다:

- **정책 A**: \\(V(2) = \\frac{-10}{1-\\gamma}\\),
  \\(V(0) = \\frac{1 + \\gamma - 7\\gamma^2}{(1-\\gamma)(1-\\gamma^2/2)}\\)
  (벨만방정식 3개를 풀어낸 것 — 다음 셀에서 잔차로 검산)
- **정책 B**: 2스텝 주기(보상 1, 0)이 반복되므로
  \\(V(0) = \\frac{1}{1-\\gamma^2}\\)

\\(\\gamma\\)를 0→1까지 스캔하면 두 곡선이 **크로스오버**한다 — 그 지점을
찾는다. 

In [ ]:
def V0_A(g):
    """정책 A(항상 A)의 상태 0 가치, 닫힌 형태."""
    V2 = -10.0 / (1.0 - g)
    V0 = (1.0 + g - 7.0 * g**2) / ((1.0 - g) * (1.0 - 0.5 * g**2))
    return V0, V2

def V0_B(g):
    """정책 B(항상 B)의 상태 0 가치, 닫힌 형태."""
    return 1.0 / (1.0 - g**2)

print(f"{'γ':>6} | {'V^A(0)':>10} | {'V^B(0)':>8} |  더 좋은 정책")
print("-" * 46)
for g in [0.0, 0.1, 0.5, 0.9, 0.99]:
    a, _ = V0_A(g)
    b = V0_B(g)
    print(f"{g:6.2f} | {a:10.4f} | {b:8.4f} |  {'A' if a > b else 'B'}")

# 본문 값 검증
a, v2 = V0_A(0.9)
assert abs(a - (-63.3613)) < 1e-3 and abs(v2 - (-100.0)) < 1e-9
assert abs(V0_B(0.9) - 1/(1-0.9**2)) < 1e-12
print(f"\nγ=0.9: V^A(0) = {a:.4f} (본문: −63.3613),  V^B(0) = {V0_B(0.9):.4f}")

# 크로스오버 γ* 찾기: 부호 변화 + 이분법
lo, hi = 1e-4, 0.9999
f = lambda g: V0_A(g)[0] - V0_B(g)
assert f(lo) > 0 and f(hi) < 0
for _ in range(80):
    mid = 0.5 * (lo + hi)
    if f(mid) > 0:
        lo = mid
    else:
        hi = mid
star = 0.5 * (lo + hi)
print(f"크로스오버 γ* = {star:.4f}  (본문: 약 0.27)")
print("γ ≲ 0.27에서는 근시안 정책 A가, γ ≳ 0.27에서는 안전 정책 B가 더 좋다")


     γ |     V^A(0) |   V^B(0) |  더 좋은 정책
----------------------------------------------
  0.00 |     1.0000 |   1.0000 |  B
  0.10 |     1.1502 |   1.0101 |  A
  0.50 |    -0.5714 |   1.3333 |  B
  0.90 |   -63.3613 |   5.2632 |  B
  0.99 |  -955.1329 |  50.2513 |  B

γ=0.9: V^A(0) = -63.3613 (본문: −63.3613),  V^B(0) = 5.2632
크로스오버 γ* = 0.2705  (본문: 약 0.27)
γ ≲ 0.27에서는 근시안 정책 A가, γ ≳ 0.27에서는 안전 정책 B가 더 좋다


### 벨만방정식 잔차 검증 (γ = 0.9)

닫힌 형태가 **정말** 벨만방정식의 해인지, 세 등호
\\(V(0) = 1 + \\gamma V(1)\\),
\\(V(1) = 2 + \\gamma (0.5 V(2) + 0.5 V(0))\\),
\\(V(2) = -10 + \\gamma V(2)\\)에 대입해 잔차를 확인한다. 

In [ ]:
g = 0.9
V0, V2 = V0_A(g)
V1 = 2 + g * (0.5 * V2 + 0.5 * V0)   # V1은 정의에서 바로 계산
r0 = abs(V0 - (1 + g * V1))
r1 = abs(V1 - (2 + g * (0.5 * V2 + 0.5 * V0)))
r2 = abs(V2 - (-10 + g * V2))
print(f"V^A = [{V0:.4f}, {V1:.4f}, {V2:.4f}]")
print(f"벨만 잔차: state0={r0:.2e}  state1={r1:.2e}  state2={r2:.2e}")
assert max(r0, r1, r2) < 1e-9
print("[OK] 세 등호 모두 성립 — 닫힌 형태가 벨만방정식의 고정점")


V^A = [-63.3613, -71.5126, -100.0000]
벨만 잔차: state0=0.00e+00  state1=0.00e+00  state2=0.00e+00
[OK] 세 등호 모두 성립 — 닫힌 형태가 벨만방정식의 고정점


## 5. CartPole 무작위 정책 베이스라인 (200 시드 × 3개 γ)

본문 실습의 코드: **같은 시드 200개**로
\\(\\gamma \\in \\{0.9, 0.95, 0.99\\}\\)의 평균 할인된 리턴을 계산한다.
버틴 **스텝 수**는 \\(\\gamma\\)와 무관한데, **리턴**은
8.62 → 13.07 → 20.77로 크게 바뀐다 — "할인율은 보상의 총량이 아니라,
언제의 보상을 얼마나 헤아리는지를 정한다"는 말의 직접 확인.
(주의: \\(env.action_space.seed(seed)\\)으로 행동 시퀀스를 **정확히 동일하게**
잡아야 γ끼리 비교가 된다.)

In [ ]:
import gymnasium as gym

env = gym.make("CartPole-v1")
BASELINE = {}
for gamma in [0.9, 0.95, 0.99]:
    returns, steps = [], []
    for seed in range(200):
        s, _ = env.reset(seed=seed)
        env.action_space.seed(seed)      # 행동 시퀀스를 γ별로 동일하게
        total, discount = 0.0, 1.0
        t = 0
        while True:
            a = env.action_space.sample()
            s, r, term, trunc, _ = env.step(a)
            total += discount * r
            discount *= gamma
            t += 1
            if term or trunc:
                break
        returns.append(total); steps.append(t)
    returns, steps = np.array(returns), np.array(steps)
    BASELINE[gamma] = (returns.mean(), returns.std(), steps.mean())
    print(f"gamma={gamma}: 무작위 정책 평균 리턴 = {returns.mean():.2f}  "
          f"(표준편차 {returns.std():.2f}, 평균 버틴 스텝 {steps.mean():.1f})  "
          f"(500스텝 다 버틸 때 최대: {(1-gamma**500)/(1-gamma):.2f})")
env.close()


gamma=0.9: 무작위 정책 평균 리턴 = 8.62  (표준편차 1.02, 평균 버틴 스텝 24.1)  (500스텝 다 버틸 때 최대: 10.00)
gamma=0.95: 무작위 정책 평균 리턴 = 13.07  (표준편차 3.11, 평균 버틴 스텝 24.1)  (500스텝 다 버틸 때 최대: 20.00)
gamma=0.99: 무작위 정책 평균 리턴 = 20.77  (표준편차 9.83, 평균 버틴 스텝 24.1)  (500스텝 다 버틸 때 최대: 99.34)


## 6. 그림 생성 (책 `kor/src/images/`에 SVG로 저장)

**그림 1** `ch03_2_discounted_return.svg` — 할인율의 세 가지 면
(본문에 삽입): (왼쪽) 감가 계수 \\(\\gamma^k\\), (가운데) \\(G_N\\)의 수렴,
(오른쪽) 예 2의 정책 가치 곡선과 크로스오버.

**그림 2** `ch03_2_gamma_policy_crossover.svg` — (왼쪽) 무작위 정책
베이스라인: 평균 리턴이 \\(\\gamma\\)에 대해 단조 증가,
(오른쪽) 10스텝 에피소드에서 "할인 안 함 / 정상 / 두 번 할인"의 리턴.


In [ ]:
# ---------- 그림 1: 할인율의 세 가지 면 (본문 개요 그림) ----------
fig, axes = plt.subplots(1, 3, figsize=(14.5, 5.0), dpi=150)

ax = axes[0]
ks = np.arange(0, 201)
for g, c in [(0.5, "#16a34a"), (0.7, "#0284c7"), (0.9, "#dc2626"), (0.99, "#7c3aed")]:
    ax.plot(ks, g**ks, color=c, lw=1.8, label=f"γ={g}")
ax.axhline(0.05, color="gray", ls="--", lw=1)
ax.text(202, 0.05, " ε = 0.05", color="gray", va="bottom", fontsize=9)
ax.set_yscale("log")
ax.set_xlim(0, 200); ax.set_ylim(1e-3, 1.2)
ax.set_xlabel("스텝 k"); ax.set_ylabel("감가 계수 γ^k (log)")
ax.set_title("감가 중시 계수: γ가 클수록\n멀리까지 0이 안 됨")
ax.legend(fontsize=9, loc="lower left")

# (가운데) G_N = 1+γ+…+γ^(N-1)의 수렴
ax = axes[1]
Ns = np.arange(1, 301)
for g, c, lim in [(0.9, "#16a34a", 10.0), (0.99, "#dc2626", 100.0)]:
    G = (1 - g**Ns) / (1 - g)
    ax.plot(Ns, G, color=c, lw=1.8, label=f"γ={g}")
    ax.axhline(lim, color=c, ls=":", lw=1)
    ax.text(301, lim, f" {lim:.0f}", color=c, va="center", fontsize=9)
ax.set_xlabel("N (스텝 수)"); ax.set_ylabel("G_N (매 스텝 R=1)")
ax.set_title("누적 리턴 G_N → 1/(1−γ)\n(γ=0.99는 100스텝에 63%만 도달)")
ax.set_xlim(1, 300)
ax.legend(fontsize=9)

# (오른쪽) 예 2: 두 정책의 상태 0 가치 vs γ
ax = axes[2]
gs = np.linspace(0.0, 0.999, 500)
VA = np.array([V0_A(g)[0] for g in gs])
VB = np.array([V0_B(g) for g in gs])
ax.plot(gs, VA, color="#16a34a", lw=1.8, label="정책 A (당장 1 → 50% 확률 −100 구덩이)")
ax.plot(gs, VB, color="#dc2626", lw=1.8, label="정책 B (0 → +1 → 0 → …)")
ax.axvline(star, color="gray", ls="--", lw=1.2)
ax.text(star + 0.01, ax.get_ylim()[1] * 0.25, f"크로스오버\nγ* ≈ {star:.2f}",
        fontsize=9, color="gray")
ax.axhline(0, color="black", lw=0.6)
ax.set_xlabel("할인율 γ"); ax.set_ylabel("상태 0의 가치 V(0)")
ax.set_title("할인율이 '최적 정책'을 바꿈")
ax.set_ylim(-110, 60)
ax.legend(fontsize=8)

fig.savefig(os.path.join(IMG, "ch03_2_discounted_return.svg"), bbox_inches="tight")
fig.subplots_adjust(left=0.05, right=0.99, top=0.94, bottom=0.12, wspace=0.35)
print("저장:", os.path.join(IMG, "ch03_2_discounted_return.svg"),
      f"({os.path.getsize(os.path.join(IMG, 'ch03_2_discounted_return.svg'))//1024} KB)")


저장: /home/smhan/book-ml/kor/src/images/ch03_2_discounted_return.svg (101 KB)


In [ ]:
# ---------- 그림 2: γ가 리턴·정책 선호에 미치는 영향 ----------
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), dpi=150)

# (왼쪽) 무작위 정책 베이스라인: 평균 리턴 vs γ (200 시드)
ax = axes[0]
gams = sorted(BASELINE)
means = [BASELINE[g][0] for g in gams]
stds  = [BASELINE[g][1] for g in gams]
ax.plot(gams, means, "o-", color="#16a34a", lw=1.8, label="평균 할인된 리턴 (200 시드)")
ax.errorbar(gams, means, yerr=stds, fmt="none", color="#16a34a", alpha=0.5, capsize=4)
max500 = [(1 - g**500) / (1 - g) for g in gams]
ax.plot(gams, max500, ":", color="gray", lw=1.4, label="상한: 500스텝 전부가 1/(1−γ)에 수렴")
ax.set_xlabel("할인율 γ"); ax.set_ylabel("평균 리턴")
ax.set_title("같은 무작위 정책, γ만 바꾸면\n리턴의 '단위'가 바뀜")
ax.legend(fontsize=8)
for g, m in zip(gams, means):
    ax.annotate(f"{m:.1f}", (g, m), textcoords="offset points", xytext=(0, 7),
                ha="center", fontsize=8, color="#16a34a")

# (오른쪽) 10스텝 에피소드(R_k=1): 정상 vs 할인 실수 두 가지
ax = axes[1]
gs2 = np.linspace(0, 1, 301)
ok   = np.array([sum(g**k for k in range(10)) for g in gs2])
none_ = np.full_like(gs2, 10.0)
dbl  = np.array([sum(g**(2*k) for k in range(10)) for g in gs2])
ax.plot(gs2, ok,   color="#16a34a", lw=2, label="정상: Σ γ^k (R_k=1, 10스텝)")
ax.plot(gs2, none_, color="#dc2626", lw=1.6, ls="--", label="할인 없이 더함: 10 (γ 무관)")
ax.plot(gs2, dbl,  color="#d97706", lw=1.6, ls="--", label="두 번 할인: Σ γ^(2k)")
for g in [0.95]:
    ax.plot([g], [sum(g**k for k in range(10))], "o", color="#16a34a", ms=7)
    ax.annotate(f"{sum(g**k for k in range(10)):.2f}", (g, sum(g**k for k in range(10))),
                textcoords="offset points", xytext=(-14, 8), fontsize=9, color="#16a34a")
    ax.plot([g], [sum(g**(2*k) for k in range(10))], "o", color="#d97706", ms=7)
    ax.annotate(f"{sum(g**(2*k) for k in range(10)):.2f}", (g, sum(g**(2*k) for k in range(10))),
                textcoords="offset points", xytext=(-14, -14), fontsize=9, color="#d97706")
ax.axvline(0.95, color="gray", ls=":", lw=1)
ax.set_xlabel("할인율 γ"); ax.set_ylabel("10스텝 리턴 G_0")
ax.set_title("γ=0.95: 정상 8.03 / 할인 없음 10 / 두 번 할인 6.58")
ax.legend(fontsize=8, loc="upper right")

fig.tight_layout()
fig.savefig(os.path.join(IMG, "ch03_2_gamma_policy_crossover.svg"), bbox_inches="tight")
print("저장:", os.path.join(IMG, "ch03_2_gamma_policy_crossover.svg"),
      f"({os.path.getsize(os.path.join(IMG, 'ch03_2_gamma_policy_crossover.svg'))//1024} KB)")


저장: /home/smhan/book-ml/kor/src/images/ch03_2_gamma_policy_crossover.svg (74 KB)


## 7. 정리

| 확인한 것 | 결론 |
|---|---|
| 직접 누적 vs 닫힌 형태 | CartPole 10스텝 에피소드에서 **8.025 = 8.025** (정확히 일치) |
| 기하급수 수렴 | \\(\\gamma=0.9\\)는 20스텝에 87.8%, \\(\\gamma=0.99\\)는 100스텝에 63.4% — γ가 클수록 더 긴 시계 |
| 유효 시계 | \\(H(0.9)=28.4\\), \\(H(0.95)=58.4\\), \\(H(0.99)=298.1\\) 스텝 (ε=0.05) |
| **할인율은 목표를 바꾼다** | 예 2 MDP에서 \\(\\gamma \\lesssim 0.27\\)이면 정책 A, \\(\\gtrsim 0.27\\)이면 정책 B가 최선 — **크로스오버 γ\\* ≈ 0.27** |
| CartPole 베이스라인 | 평균 리턴 **8.62 → 13.07 → 20.77** (γ: 0.9→0.99), 버틴 스텝 수는 거의 불변 — 리턴은 **γ마다 단위가 다른 화폐** |
| 할인 실수 | 10스텝(γ=0.95): 정상 8.03, 할인 없음 10, **두 번 할인 6.58** — "리턴이 이상하면 discount 곱셈을 grep" |

이 \\(G_t = \\sum_{k=0}^{\\infty} \\gamma^k R_{t+k}\\)가 바로 3.3절의
**가치함수** \\(V^{\\pi}(s) = \\mathbb{E}[G_t \\mid s_t = s]\\)가 기대를
취하는 리턴 그 자체다. 다음 노트북(3.3)에서는 이 리턴을 "한 스텝짜리
재귀식" — 벨만방정식 — 으로 다시 쓴다.
